# 05 Build Ablation Datasets

Build YOLO-ready datasets for the four PPE ablation experiments using the current four-class taxonomy: `person`, `helmet`, `vest`, and `cleaning_coverall`.


## Purpose

Notebook 05 copies already-prepared data into experiment folders. It does not split data, augment data, or train models.

The source contract is:

- Original train/validation/test splits come from `data/generated/splits/`.
- Offline augmented training samples come from `data/generated/augmented/`.
- Experiment datasets are written to `data/generated/experiments/`.

Validation and test folders are copied identically into every experiment. Only the training folder changes between experiments.


## Ablation Design

- `exp_A_original_only`: original train only; online augmentation is off later.
- `exp_B_online_aug`: original train only; online augmentation is on later.
- `exp_C_offline_aug`: original train plus offline augmented train; online augmentation is off or minimal later.
- `exp_D_full_pipeline`: original train plus offline augmented train; online augmentation is on later.

The notebook intentionally does not save extra figures or detailed class-distribution CSVs. The compact summary already includes all four class counts, including `num_cleaning_coverall`.


## Setup

Find the `v2_pipeline` root, add it to `sys.path`, and import only the small set of helpers needed for this notebook.


In [1]:
from pathlib import Path
import sys

import pandas as pd
import yaml
from IPython.display import display


def find_v2_root(start: Path) -> Path:
    """Find the local v2_pipeline root from any notebook launch directory."""
    # Jupyter may start from the notebook folder, the repo root, or a VS Code
    # workspace folder. Walk upward first so the notebook stays portable.
    for candidate in (start, *start.parents):
        if candidate.name == "v2_pipeline" and (candidate / "src").exists():
            return candidate

        # This branch lets the notebook run from the repository root in VS Code.
        nested = candidate / "ppe-detection" / "v2_pipeline"
        if nested.exists() and (nested / "src").exists():
            return nested

    raise RuntimeError("Could not find v2_pipeline root")


V2_ROOT = find_v2_root(Path.cwd().resolve())

# Add the pipeline root once so imports like `from src.dataset...` work no
# matter where Jupyter launched the kernel from.
if str(V2_ROOT) not in sys.path:
    sys.path.insert(0, str(V2_ROOT))

# Heavy copying, counting, YAML writing, and integrity checks stay in src.
# The notebook only decides paths, calls the builder, and displays compact tables.
from src.dataset.build_experiment_sets import build_ablation_datasets

V2_ROOT

WindowsPath('C:/Github/smart-factory-safety-monitoring-system/ppe-detection/v2_pipeline')

The reusable copying and integrity logic lives in `src/dataset/build_experiment_sets.py`. Keeping it there makes this notebook easier to read and safer to rerun after code changes.


## Load Configuration

Read class names and generated-data paths from config files. No absolute local path is hardcoded in this notebook.


In [2]:
dataset_config_path = V2_ROOT / "configs" / "dataset_config.yaml"
class_names_path = V2_ROOT / "configs" / "class_names.yaml"

# Load config files once near the top. The notebook should follow the same
# path contract as the rest of the pipeline instead of hardcoding folders.
with dataset_config_path.open("r", encoding="utf-8") as file_handle:
    dataset_config = yaml.safe_load(file_handle)

with class_names_path.open("r", encoding="utf-8") as file_handle:
    class_config = yaml.safe_load(file_handle)

# Normalize YAML keys to integers so class 3 is handled exactly like classes 0-2.
class_names = {int(class_id): name for class_id, name in class_config["names"].items()}

# Notebook 03 and 04 write generated data under this root. Notebook 05 only copies
# from those generated folders into experiment-specific folders.
splits_dir = V2_ROOT / dataset_config["generated"]["splits"]
augmented_dir = V2_ROOT / dataset_config["generated"]["augmented"]
experiments_dir = V2_ROOT / dataset_config["generated"]["experiments"]
reports_dir = V2_ROOT / dataset_config["reports"]["experiments"]

# Show the resolved paths before copying anything. This is the easiest place to
# catch a stale config or a notebook launched from the wrong repository.
path_summary = pd.DataFrame(
    [
        {"setting": "splits_dir", "value": str(splits_dir)},
        {"setting": "augmented_dir", "value": str(augmented_dir)},
        {"setting": "experiments_dir", "value": str(experiments_dir)},
        {"setting": "reports_dir", "value": str(reports_dir)},
        {"setting": "class_names", "value": class_names},
    ]
)
display(path_summary)

,setting,value
0,splits_dir,C:\Github\smart-factory-safety-monitoring-syst...
1,augmented_dir,C:\Github\smart-factory-safety-monitoring-syst...
2,experiments_dir,C:\Github\smart-factory-safety-monitoring-syst...
3,reports_dir,C:\Github\smart-factory-safety-monitoring-syst...
4,class_names,"{0: 'person', 1: 'helmet', 2: 'vest', 3: 'clea..."


Class `3 = cleaning_coverall` is part of the same YOLO label file as the other classes. No special file handling is required; the builder counts it from labels and carries it through every experiment copy.


## Source Counts

Check the original splits and flat offline augmentation folder before building experiments. These counts are only a quick sanity check; the builder still performs its own validation before copying.


In [3]:
VALID_IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}


def count_files(directory: Path, suffixes: set[str]) -> int:
    """Count visible files with one of the expected suffixes."""
    if not directory.exists():
        return 0

    # Hidden files such as `.gitkeep` are intentionally ignored so placeholder
    # files do not look like missing or invalid dataset samples.
    return sum(
        1
        for path in directory.iterdir()
        if path.is_file()
        and not path.name.startswith(".")
        and path.suffix.lower() in suffixes
    )


# These counts are a fast pre-flight view only. The builder still verifies that
# each split contains matched image-label pairs before copying.
split_counts = pd.DataFrame(
    [
        {
            "source": f"splits/{split}",
            "num_images": count_files(
                splits_dir / split / "images", VALID_IMAGE_EXTENSIONS
            ),
            "num_labels": count_files(splits_dir / split / "labels", {".txt"}),
        }
        for split in ["train", "val", "test"]
    ]
)

# Offline augmentation is now a flat train-only folder. We do not scan old
# per-augmentation subfolders here because Notebook 04 writes into one shared
# generated location.
augmented_counts = pd.DataFrame(
    [
        {
            "source": "augmented/train_only",
            "num_images": count_files(augmented_dir / "images", VALID_IMAGE_EXTENSIONS),
            "num_labels": count_files(augmented_dir / "labels", {".txt"}),
        }
    ]
)

display(pd.concat([split_counts, augmented_counts], ignore_index=True))

if augmented_counts["num_images"].sum() == 0:
    print(
        "Warning: no offline augmented images found. Experiments C and D will still be built from original train data only."
    )

,source,num_images,num_labels
0,splits/train,596,596
1,splits/val,104,104
2,splits/test,42,42
3,augmented/train_only,298,298


The training split already includes the open-source data from Notebook 03. Notebook 05 therefore does not read a separate open-source folder again; that avoids duplicating public samples in the experiment datasets.


## Build Experiment Datasets

Set `OVERWRITE_EXISTING_EXPERIMENTS=True` only when you intentionally want to regenerate the experiment folders. With the default `False`, the notebook stops safely if files already exist.


In [4]:
# Keep this False for normal use. If experiment folders already contain files,
# the builder raises FileExistsError instead of silently mixing old and new data.
# Set True only when you deliberately want to regenerate all experiment folders.
OVERWRITE_EXISTING_EXPERIMENTS = True

# The builder returns exactly the three audit tables this notebook needs:
# 1. row-level copy report
# 2. compact split/class summary, including cleaning_coverall
# 3. integrity warnings for missing pairs, conflicts, or val/test drift
copy_report, summary_df, warnings_df = build_ablation_datasets(
    splits_dir=splits_dir,
    augmented_dir=augmented_dir,
    experiments_dir=experiments_dir,
    class_names=class_names,
    output_yaml_dir=V2_ROOT,
    overwrite=OVERWRITE_EXISTING_EXPERIMENTS,
)

print("Ablation dataset build complete.")

# Show only a small preview. The full copy audit is saved below, but printing the
# whole table can make the notebook noisy on real datasets.
display(copy_report.head(10))

Ablation dataset build complete.


,experiment,split,source_type,original_image_path,original_label_path,copied_image_path,copied_label_path,status,notes
0,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
1,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
2,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
3,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
4,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
5,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
6,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
7,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
8,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,
9,exp_A_original_only,train,original_train,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,C:\Github\smart-factory-safety-monitoring-syst...,copied,


## Dataset YAML Files

Each experiment receives one Ultralytics dataset YAML at the `v2_pipeline` root. The YAML paths point to `data/generated/experiments/...`.


In [5]:
dataset_yaml_paths = [
    V2_ROOT / "data_exp_A_original_only.yaml",
    V2_ROOT / "data_exp_B_online_aug.yaml",
    V2_ROOT / "data_exp_C_offline_aug.yaml",
    V2_ROOT / "data_exp_D_full_pipeline.yaml",
]

# These YAML files are consumed by the training notebooks. Their `path` values
# point to `data/generated/experiments/...`, while online augmentation remains a
# training-time argument rather than a dataset-file setting.
yaml_status = pd.DataFrame(
    [
        {"yaml_file": path.name, "exists": path.exists(), "path": str(path)}
        for path in dataset_yaml_paths
    ]
)
display(yaml_status)

,yaml_file,exists,path
0,data_exp_A_original_only.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...
1,data_exp_B_online_aug.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...
2,data_exp_C_offline_aug.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...
3,data_exp_D_full_pipeline.yaml,True,C:\Github\smart-factory-safety-monitoring-syst...


## Save Compact Reports

Notebook 05 saves only the reports needed to audit the copied datasets. It avoids extra figures and redundant CSVs so the repository does not fill up with noisy artifacts.


In [6]:
# Create the reports folder only when this notebook runs. Generated reports are
# ignored by git, so they are for local audit/review rather than source control.
reports_dir.mkdir(parents=True, exist_ok=True)

# Keep artifacts intentionally small: one detailed copy audit, one compact
# summary, and one warning table. Class counts are already inside the summary,
# so we do not save a separate class-distribution CSV here.
copy_report.to_csv(reports_dir / "ablation_dataset_report.csv", index=False)
summary_df.to_csv(reports_dir / "ablation_dataset_summary.csv", index=False)
warnings_df.to_csv(reports_dir / "ablation_integrity_warnings.csv", index=False)

print(f"Saved compact experiment reports to: {reports_dir}")

Saved compact experiment reports to: C:\Github\smart-factory-safety-monitoring-system\ppe-detection\v2_pipeline\reports\experiments


## Summary

Review the generated experiment sizes and class counts. Validation and test counts should match across all four experiments.


In [7]:
# Full summary: all experiments, all splits, all four classes.
display(summary_df)

# Training split is where the ablation conditions differ. A/B should match each
# other, C/D should match each other, and C/D should be larger when offline
# augmentation exists.
train_summary = summary_df.loc[summary_df["split"].eq("train")].copy()
display(
    train_summary[
        [
            "experiment",
            "num_images",
            "num_labels",
            "num_objects",
            "num_person",
            "num_helmet",
            "num_vest",
            "num_cleaning_coverall",
            "notes",
        ]
    ]
)

,experiment,split,num_images,num_labels,num_no_object_images,num_objects,num_person,num_helmet,num_vest,num_cleaning_coverall,notes
0,exp_A_original_only,train,596,596,0,4757,1955,1431,1371,0,train uses original split only; online augment...
1,exp_A_original_only,val,104,104,0,733,293,199,241,0,fixed split
2,exp_A_original_only,test,42,42,0,165,84,39,42,0,fixed split
3,exp_B_online_aug,train,596,596,0,4757,1955,1431,1371,0,train uses original split only; online augment...
4,exp_B_online_aug,val,104,104,0,733,293,199,241,0,fixed split
5,exp_B_online_aug,test,42,42,0,165,84,39,42,0,fixed split
6,exp_C_offline_aug,train,894,894,0,7294,3012,2160,2122,0,train uses original plus offline augmented sam...
7,exp_C_offline_aug,val,104,104,0,733,293,199,241,0,fixed split
8,exp_C_offline_aug,test,42,42,0,165,84,39,42,0,fixed split
9,exp_D_full_pipeline,train,894,894,0,7294,3012,2160,2122,0,train uses original plus offline augmented sam...


,experiment,num_images,num_labels,num_objects,num_person,num_helmet,num_vest,num_cleaning_coverall,notes
0,exp_A_original_only,596,596,4757,1955,1431,1371,0,train uses original split only; online augment...
3,exp_B_online_aug,596,596,4757,1955,1431,1371,0,train uses original split only; online augment...
6,exp_C_offline_aug,894,894,7294,3012,2160,2122,0,train uses original plus offline augmented sam...
9,exp_D_full_pipeline,894,894,7294,3012,2160,2122,0,train uses original plus offline augmented sam...


## Integrity Warnings

Warnings guide review before training. For very small datasets, a missing class in validation or test can be expected, but mismatched image/label counts or differing validation/test signatures should be fixed before Notebook 06.


In [8]:
# Review warnings before training. Some class-missing warnings can be normal for
# small validation/test sets, but mismatched labels or differing val/test splits
# should be fixed before Notebook 06.
if warnings_df.empty:
    print("No integrity warnings found.")
else:
    display(warnings_df)

,experiment,split,warning_type,details
0,exp_A_original_only,train,class_missing_from_split,3: cleaning_coverall
1,exp_A_original_only,val,class_missing_from_split,3: cleaning_coverall
2,exp_A_original_only,test,class_missing_from_split,3: cleaning_coverall
3,exp_B_online_aug,train,class_missing_from_split,3: cleaning_coverall
4,exp_B_online_aug,val,class_missing_from_split,3: cleaning_coverall
5,exp_B_online_aug,test,class_missing_from_split,3: cleaning_coverall
6,exp_C_offline_aug,train,class_missing_from_split,3: cleaning_coverall
7,exp_C_offline_aug,val,class_missing_from_split,3: cleaning_coverall
8,exp_C_offline_aug,test,class_missing_from_split,3: cleaning_coverall
9,exp_D_full_pipeline,train,class_missing_from_split,3: cleaning_coverall
